In [1]:
import pypeline
import numpy as np
from glob import glob
import os
import sys
import mne_bids
from contextlib import contextmanager
import mne
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import shutil
from pathlib import Path
%load_ext autoreload

%autoreload 2

# Preprocessing subject 011.
Their eyetracker switched the sampling rate from 1000Hz to 250Hz partway through the session.
The solution here is to split both the EEG and eyes into 2 sections, upsample the eyes back to 500Hz, and then concatenate everything

In [2]:
EXPERIMENT_NAME = "banyan"  # name of the experiment
parent_dir = "../raw_data"  # directory where your raw data (folder containing brainvision, eyetracking asc, and behavior is stored)
data_dir = "../data"  # where to output data
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
file_prefix = ""  # prefix to your vhdr files. Assuming it is in the format [prefix][number]

overwrite_subs = False  # if you want to overwrite the data for a subject, set to True
subject_dirs = ["011"]  # if you want to analyze a specific subset of subjects
if len(subject_dirs) == 0:
    subject_dirs = sorted(glob("*", root_dir=parent_dir))
if not overwrite_subs:
    subject_dirs = [sub for sub in subject_dirs if sub not in [f[4:] for f in glob("sub-*", root_dir=data_dir)]]
subject_dirs = ["011"]  # if you want to analyze a specific subset of subjects

print("Subject to analyze: ", subject_dirs)


TRIAL_START_TIME = -0.45  # epoch start before your designated timelock code
TRIAL_END_TIME = 1.4
BASELINE_TIME = (-0.35, 0)  # time for baseline correction

SRATE = 500  # hz, will resample if different from the raw data srate
FILTER_FREQS = (None, 50)  # None to not do one of the filtering steps

LINEAR_R2 = 0.3

event_dict = {  # this should be a dict of names of ALL the event codes that appear
    "trl_start": 1,
    "attn/ss2/abs": 12,
    "attn/ss4/abs": 14,
    "attn/ss6/abs": 16,
    "attn/ss8/abs": 18,
    "attn/ss2/placeholder": 22,
    "attn/ss4/placeholder": 24,
    "attn/ss6/placeholder": 26,
    "attn/ss8/placeholder": 28,
    "wm/ss2/abs": 32,
    "wm/ss4/abs": 34,
    "wm/ss6/abs": 36,
    "wm/ss8/abs": 38,
    "wm/ss2/placeholder": 42,
    "wm/ss4/placeholder": 44,
    "wm/ss6/placeholder": 46,
    "wm/ss8/placeholder": 48,
    "delay_start": 2,
    "attention_probe": 3,  # if probe attention, the code seq is 2 3 2 4
    "delay_end": 4,
    "response": 5,
}
# event_dict is a list of name: number pairings for all the TRIAL event codes
# event_code_dict: a dict of code: sequence pairings for each trial
# so, if you have a trial with fixation (1) -> SS2 stimulus (12) -> delay (3) -> test (4), that you want to map to code  12:
# {12 : [1,12,3,4]}... and so on
event_code_dict = {}  # define event codes based on sequence

for key, ev in event_dict.copy().items():
    if ev > 10:
        event_dict.update({key + "/PROBE": ev + 1})  # add in keys for targets
        event_code_dict.update({ev: [1, ev, 2, 4, 4]})
        event_code_dict.update({ev + 1: [1, ev, 2, 3, 2, 4, 4]})


POSITION_TO_TIMELOCK = 1  # which position (IN THE LIST ABOVE) to timelock to. Should be the one which indicates the condition.

EYEKEYWORD = "SYNC"

# edge case when we forgot to start the recording, manually drop certain trials
# Must be in the form of {'subject number':[list of ints]}
EEG_TRIALS_DROP = {}
EYE_TRIALS_DROP = {"010": [0, 1, 2, 3]}  # edge case when we forgot to start the recording, drop certain trials
# channels to ignore when rejecting trials. Will also be excluded from analysis.
SUB_IGNORE_CHANS = {}

NO_EYES = []  # subjects with no eye data
DROP_CHANNELS = (
    []
)  # channels to delete from the dataset entirely. Recommendation is to leave this blank and instead set REJ_TRIALS_IGNORE later on

Subject to analyze:  ['011']


In [3]:
pre = pypeline.Preprocess(
    data_dir=data_dir,
    root_dir=parent_dir,
    experiment_name=EXPERIMENT_NAME,
    srate=SRATE,
    trial_start=TRIAL_START_TIME,
    trial_end=TRIAL_END_TIME,
    event_dict=event_dict,
    event_code_dict=event_code_dict,
    timelock_ix=POSITION_TO_TIMELOCK,
    baseline_time=BASELINE_TIME,
    rejection_time=None,
    reject_between_codes=[1, 4],
    no_et_spaces=False,
    drop_channels=DROP_CHANNELS,
    filter_freqs=FILTER_FREQS,
)

In [4]:
@contextmanager
def redirect_stdout(new_stdout):  # writes the output to a log file
    save_stdout = sys.stdout
    save_stderr = sys.stderr
    sys.stdout = new_stdout
    sys.stderr = sys.stdout
    try:
        yield None
    finally:
        sys.stdout = save_stdout
        sys.stderr = save_stderr


# with open('preprocessing_log.txt','a+') as f:
#     with redirect_stdout(f):

print("\n\n\n##########################\n" + "STARTING PREPROCESSING RUN\n" + "##########################\n\n\n")
print(f'Run started at {datetime.now().strftime("%H:%M:%S")}')
for subject_number in subject_dirs:

    print(
        "\n\n#############################\n"
        + f"## STARTING NEW SUBJECT {subject_number} ##\n"
        + "#############################\n"
    )

    #####################
    #### IMPORT DATA ####
    #####################

    # import into the RAW bids dataset
    eeg, eeg_events = pre.import_eeg(subject_number, overwrite=True)
    reref_index = mne.pick_channels(eeg.ch_names, ["TP9"])
    eeg.load_data().apply_function(
        pre.rereference_to_average, picks=["eeg"], reref_values=np.squeeze(eeg.get_data()[reref_index])
    )
    eeg.filter(*pre.filter_freqs, n_jobs=-1)

    pre.import_behavior(subject_number)

    ########################################
    #### PREPROCESS EEG AND MAKE EPOCHS ####
    ########################################

    subject_number = subject_dirs[0]
    keyword = "SYNC"
    subject_dir = os.path.join(pre.root_dir, subject_number)
    path = mne_bids.path.BIDSPath(
        subject=subject_number,
        task=pre.experiment_name,
        root=pre.data_dir,
        datatype="eyetracking",
        suffix="eyetracking",
        extension=".asc",
        check=False,
    )
    path.mkdir()

    # COPY MAIN ASC FILE (with spaces removed)

    asc_file = sorted(glob("*.asc", root_dir=subject_dir))

    for ifile, file in enumerate(asc_file):
        file = os.path.join(subject_dir, file)

        ascpath = path.copy().update(split=ifile + 1)

        if pre.no_et_spaces:
            shutil.copy2(file, ascpath.fpath)
        else:
            pre._remove_eyetrack_spaces(file, ascpath.fpath)

    raws = []

    file = asc_file[0]
    file = os.path.join(subject_dir, file)
    ascpath = path.copy().update(split=1)
    eye0 = mne.io.read_raw_eyelink(ascpath.fpath, create_annotations=["blinks", "messages"])
    ascpath = path.copy().update(split=2)
    eye1 = mne.io.read_raw_eyelink(ascpath.fpath, create_annotations=["blinks", "messages"])

    keyword = "" if keyword is None else keyword
    regexp = f"^{keyword}(?![Bb][Aa][Dd]|[Ee][Dd][Gg][Ee]).*$"
    eye_events0, et_event_dict0 = mne.events_from_annotations(eye0, regexp=regexp)
    eye_events1, et_event_dict1 = mne.events_from_annotations(eye1, regexp=regexp)
    et_event_dict = et_event_dict0 | et_event_dict1

    et_events_dict_convert = {}
    for k, v in et_event_dict.items():
        new_k = int(k.split(" ")[-1])
        et_events_dict_convert[v] = new_k

    et_events_converted = eye_events0.copy()
    for code in et_events_dict_convert.keys():
        et_events_converted[:, 2][eye_events0[:, 2] == code] = et_events_dict_convert[code]
    eye_events0 = et_events_converted.copy()
    et_events_converted = eye_events1.copy()
    for code in et_events_dict_convert.keys():
        et_events_converted[:, 2][eye_events1[:, 2] == code] = et_events_dict_convert[code]
    eye_events1 = et_events_converted.copy()

    eeg_trials_drop = []
    eye_trials_drop = []
    # get our events list
    unmatched_codes = list(set(eeg_events["value"].unique()) ^ (set(eye_events0[:, 2]) | set(eye_events1[:, 2])))
    # convert event dataframe to mne format (array of sample, duration, value)
    eeg_events = pre._convert_bids_events(eeg_events)

    eeg_events = eeg_events[~np.isin(eeg_events, unmatched_codes).any(axis=1)]
    eye_events0 = eye_events0[~np.isin(eye_events0, unmatched_codes).any(axis=1)]
    eye_events1 = eye_events1[~np.isin(eye_events1, unmatched_codes).any(axis=1)]
    eeg_events_lock, eeg_events = pre._filter_events(eeg_events)  # get all events
    eye_events0, _ = pre._filter_events(eye_events0)  # only get events we timelock to
    eye_events1, _ = pre._filter_events(eye_events1)  # only get events we timelock to

    extra_eeg_trials, extra_eye_trials = pre._check_event_desyncs(
        eeg_events_lock[:, 2], np.concatenate((eye_events0, eye_events1))[:, 2]
    )
    eeg_trials_drop = list(set(eeg_trials_drop) | set(extra_eeg_trials))
    eye_trials_drop = list(set(eye_trials_drop) | set(extra_eye_trials))

    metadata, metadata_events = pre._make_metadata_from_events(eeg_events, eeg.info["sfreq"])  # only from EEG data

    eeg_epochs = mne.Epochs(
        eeg,
        metadata_events,
        pre.event_dict,
        tmin=pre.trial_start_t,
        tmax=pre.trial_end_t,
        on_missing="ignore",
        metadata=metadata,
        baseline=pre.baseline_time,
        preload=True,
        decim=2,  # downsample to 500
    ).drop(
        eeg_trials_drop
    )  # set up epochs object
    if pre.drop_channels is not None:
        eeg_epochs = eeg_epochs.drop_channels(pre.drop_channels)

    eeg_epochs0 = eeg_epochs.copy()[0 : len(eye_events0)]
    eeg_epochs1 = eeg_epochs.copy()[len(eye_events0) :]

    # this one is at 500, so decimate accordingly
    eye_epochs0 = mne.Epochs(
        eye0,
        eye_events0,
        pre.event_dict,
        tmin=pre.trial_start_t,
        tmax=pre.trial_end_t,
        on_missing="ignore",
        baseline=pre.baseline_time,
        reject=None,
        flat=None,
        reject_by_annotation=False,
        preload=True,
        decim=2,  # downsample to 500
    )

    eye_epochs0.drop_channels(["DIN"])

    eye_epochs1 = mne.Epochs(
        eye1,
        eye_events1,
        pre.event_dict,
        tmin=pre.trial_start_t,
        tmax=pre.trial_end_t,
        on_missing="ignore",
        baseline=pre.baseline_time,
        reject=None,
        flat=None,
        reject_by_annotation=False,
        preload=True,
    )
    eye_epochs1.drop_channels(["DIN"])
    eye_epochs1.apply_function(lambda x: np.nan_to_num(x, nan=0), picks="eyegaze")
    eye_epochs1.resample(
        500
    )  # upsample to 500, decimating is preferred so doing that when downsampling, but can't do that here

    eeg_epochs0.add_channels([eye_epochs0], force_update_info=True)
    eeg_epochs1.add_channels([eye_epochs1], force_update_info=True)

    epochs = mne.concatenate_epochs([eeg_epochs0, eeg_epochs1])

    ###############################
    #### DO ARTIFACT REJECTION ####
    ###############################

    p2p = pre.artreject_slidingP2P(
        epochs, rejection_criteria={"eeg": 100e-6, "eog": 200}, win=200, win_step=100
    )  # peak to peak in the window
    saccades = pre.artreject_step(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(0.5), "eog": 50}, win=80, win_step=10
    )  # saccades in EOG or eye tracking
    steps = pre.artreject_step(
        epochs, rejection_criteria={"eeg": 60e-6}, win=250, win_step=20
    )  # steps (saccade like) in EEG

    absolute_value = pre.artreject_value(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(1), "eeg": 100e-6, "eog": 300}
    )  # absolute value rejection
    linear_fit = pre.artreject_linear(epochs)  # linear fit (drift) rejection
    flatline = pre.artreject_flatline(
        epochs,
        rejection_criteria={"eeg": 0, "eog": 0, "eyegaze": 0},
        flatline_duration=200,
    )  # check for flatlines

    # combine rejection reasons
    rej_electrodes = p2p | saccades | steps | absolute_value | linear_fit | flatline
    rej_reasons = np.char.array(
        np.full(rej_electrodes.shape, "", dtype="<U30")
    )  # NOTE: dtype is important, must be >= the max possible str length
    rej_reasons[p2p] = "P2P "
    rej_reasons[saccades] = rej_reasons[saccades] + "SAC "
    rej_reasons[steps] = rej_reasons[steps] + "STEP "
    rej_reasons[absolute_value] = rej_reasons[absolute_value] + "ABS "
    rej_reasons[linear_fit] = rej_reasons[linear_fit] + "LIN "
    rej_reasons[flatline] = rej_reasons[flatline] + "FLAT "

    rej_counts = lambda x: f"{x.any(1).sum()} ({round(x.any(1).sum() / x.shape[0] * 100,1)}%)"
    print(
        (
            f"Rejected {rej_electrodes.any(1).sum()} trials ({round(rej_electrodes.any(1).sum() / rej_electrodes.shape[0] * 100,1)}%) for the following reasons:\n"
            f"Peak to peak amplitude: {rej_counts(p2p)}\n"
            f"Saccades: {rej_counts(saccades)}\n"
            f"Steps: {rej_counts(steps)}\n"
            f"Absolute value: {rej_counts(absolute_value)}\n"
            f"Linear fit: {rej_counts(linear_fit)}\n"
            f"Flatline: {rej_counts(flatline)}\n"
        )
    )

    print(
        "Worst electrodes by count:\n"
        + "\n".join(
            [f"{epochs.ch_names[i]}: {rej_electrodes[:,i].sum()}" for i in np.argsort(rej_electrodes.sum(0))[::-1][0:5]]
        )
    )

    #################################
    #### SAVE DATA AS DERIVATIVE ####
    #################################

    pre.save_all_data(subject_number, epochs, rej_reasons)
print(mne_bids.make_report(data_dir))




##########################
STARTING PREPROCESSING RUN
##########################



Run started at 11:09:20


#############################
## STARTING NEW SUBJECT 011 ##
#############################

Extracting parameters from ../raw_data/011/SCR_011.vhdr...
Setting channel info structure...


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Channels contain different highpass filters. Lowest (weakest) filter setting (0.00 Hz) will be stored.
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (500.00 Hz, Nyquist limit) will be stored.
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Not setting positions of 3 eog/misc channels found in montage:
['

Used Annotations descriptions: [np.str_('New Segment/'), np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 14'), np.str_('Stimulus/S 16'), np.str_('Stimulus/S 18'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S 24'), np.str_('Stimulus/S 26'), np.str_('Stimulus/S 28'), np.str_('Stimulus/S 32'), np.str_('Stimulus/S 34'), np.str_('Stimulus/S 36'), np.str_('Stimulus/S 38'), np.str_('Stimulus/S 42'), np.str_('Stimulus/S 44'), np.str_('Stimulus/S 46'), np.str_('Stimulus/S 48')]


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:196: RuntimeWarning: Converting data files to BrainVision format
  mne_bids.write_raw_bids(
/opt/homebrew/Caskroom/miniforge/base/envs/preproc_pype/lib/python3.12/site-packages/pybv/io.py:690: UserWarning: Encountered unsupported non-voltage units: n/a
Note that the BrainVision format specification supports only µV.
  warn(msg)


Writing '../data/sub-011/eeg/sub-011_task-banyan_eeg.json'...
Reading 0 ... 12164259  =      0.000 ... 12164.259 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 265 samples (0.265 s)



...

/var/folders/g5/_50pp0f95mx6v6d1_3b33r700000gn/T/ipykernel_40109/1109346040.py:145: RuntimeWarning: The measurement information indicates a low-pass frequency of 500.0 Hz. The decim=2 parameter will result in a sampling frequency of 500.0 Hz, which can cause aliasing artifacts.
  eye_epochs0 = mne.Epochs(


Adding metadata with 20 columns
1599 matching events found
Applying baseline correction (mode: mean)
Rejected 478 trials (29.9%) for the following reasons:
Peak to peak amplitude: 437 (27.3%)
Saccades: 55 (3.4%)
Steps: 1 (0.1%)
Absolute value: 9 (0.6%)
Linear fit: 0 (0.0%)
Flatline: 0 (0.0%)

Worst electrodes by count:
Fp1: 391
Fp2: 66
xpos_right: 48
xpos_left: 48
ypos_left: 10


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:1120: RuntimeWarning: This filename (../data/derivatives/sub-011/eeg/sub-011_task-banyan_desc-preprocessed_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs.save(path.fpath, overwrite=True)


Summarizing participants.tsv ../data/participants.tsv...
Summarizing scans.tsv files [PosixPath('../data/sub-021/sub-021_scans.tsv'), PosixPath('../data/sub-026/sub-026_scans.tsv'), PosixPath('../data/sub-019/sub-019_scans.tsv'), PosixPath('../data/sub-010/sub-010_scans.tsv'), PosixPath('../data/sub-017/sub-017_scans.tsv'), PosixPath('../data/sub-028/sub-028_scans.tsv'), PosixPath('../data/sub-016/sub-016_scans.tsv'), PosixPath('../data/sub-029/sub-029_scans.tsv'), PosixPath('../data/sub-011/sub-011_scans.tsv'), PosixPath('../data/sub-027/sub-027_scans.tsv'), PosixPath('../data/sub-018/sub-018_scans.tsv'), PosixPath('../data/sub-020/sub-020_scans.tsv'), PosixPath('../data/sub-002/sub-002_scans.tsv'), PosixPath('../data/sub-005/sub-005_scans.tsv'), PosixPath('../data/sub-033/sub-033_scans.tsv'), PosixPath('../data/sub-034/sub-034_scans.tsv'), PosixPath('../data/sub-032/sub-032_scans.tsv'), PosixPath('../data/sub-004/sub-004_scans.tsv'), PosixPath('../data/sub-003/sub-003_scans.tsv'), Po

/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:1122: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  np.save(path.fpath, epochs.get_data())


In [5]:
%matplotlib qt
# sub = input('Enter subject number: ') # you can also just set this to a string
sub = "011"
REJ_CHANNELS_IGNORE=['StimTrak', 'HEOG','VEOG','TP9', 'Fp1', 'Fp2'] # exclude 'Fp1' and 'Fp2' here if you want to keep trials where they are noisy, and drop them before analysis later


viz = pypeline.Visualizer(sub,
                            parent_dir = data_dir,
                            load_flags = False,  # we're going to rebuild our rejection flags every time
                            port_codes_show=None,
                            experiment_name=EXPERIMENT_NAME,
                            srate=SRATE,
                            rejection_time=[None, None],
                            downscale={'eyegaze':1e-6,'misc':1,'eeg':1,'eog':1e-6}, # convert to equivalent units (probably uV)
                            channels_drop=['pupil_left','pupil_right'],
                            channels_ignore=REJ_CHANNELS_IGNORE + SUB_IGNORE_CHANS.get(sub, []))


rejection_sums = viz.rej_chans.sum(axis=0)
sort_ix = np.argsort(rejection_sums)[::-1]

for ichan,chan in enumerate(viz.chan_labels[sort_ix]):
    if rejection_sums[sort_ix][ichan] > 0:
        print(chan,rejection_sums[sort_ix][ichan]) 
viz.preprocess_data_for_plot()
viz.open_figure()

Reading /Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/../data/derivatives/sub-011/eeg/sub-011_task-banyan_desc-preprocessed_eeg.fif ...
    Found the data of interest:
        t =    -450.00 ...    1400.00 ms
        0 CTF compensation matrices available


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/visualizer.py:56: RuntimeWarning: This filename (../data/derivatives/sub-011/eeg/sub-011_task-banyan_desc-preprocessed_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.epochs_obj = mne.read_epochs(self.data_path.fpath)


Adding metadata with 20 columns
1599 matching events found
No baseline correction applied
0 projection items activated
xpos_left 48
xpos_right 48
ypos_right 10
ypos_left 10
F8 6
Cz 2


62/1599 trials rejected. Saving annotations as "../data/derivatives/sub-011/eeg/sub-011_task-banyan_desc-preprocessed_rejection_flags.npy"
